# ACIS Auto Insurance: A/B Hypothesis Testing

This notebook statistically validates key risk drivers to establish a mathematical foundation for ACIS's new segmentation and pricing strategy.

### Actuarial KPI Definitions
* **Claim Frequency** (Categorical): Proportion of policies with at least one claim (`has_claim`).
* **Claim Severity** (Numerical): Average claim amount given a claim occurred (`totalclaims > 0`).
* **Margin** (Numerical): $\text{TotalPremium} - \text{TotalClaims}$.

### Null Hypotheses ($H_0$)
1. **$H_0^1$ (Provinces)**: No risk differences (Claim Frequency) across provinces.
2. **$H_0^2$ (Zip Codes - Risk)**: No risk differences (Claim Frequency) between zip codes.
3. **$H_0^3$ (Zip Codes - Profit)**: No significant margin difference between zip codes.
4. **$H_0^4$ (Gender)**: No significant risk difference between Women and Men.

*Decision Rule: Reject $H_0$ if $p < 0.05$.*


In [1]:
# Ingestion & Preprocessing
import pandas as pd
import numpy as np
import sys
import os

# Allow importing from the src directory
sys.path.append(os.path.abspath('../'))
from src.hypothesis_tests import run_chi2_test, run_ttest

# Load the cleaned version of the dataset tracked by DVC (separated by '|')
df = pd.read_csv('../data/insurance_data.csv', sep='|', low_memory=False)

# Actuarial KPI Definitions:
df['has_claim'] = np.where(df['totalclaims'] > 0, 1, 0)

df['margin'] = df['totalpremium'] - df['totalclaims']

print(f"Dataset successfully loaded. Shape: {df.shape}")
print(f"Total claims observed: {df['has_claim'].sum()} policies")


Dataset successfully loaded. Shape: (1000098, 51)
Total claims observed: 2788 policies


In [3]:
# Hypothesis 1: H0: There are no risk differences across provinces.
print("--- Claim Frequency by Province ---")
summary = df.groupby('province').agg(
    total_policies=('has_claim', 'count'),
    total_claims=('has_claim', 'sum')
)
summary['claim_rate (%)'] = (summary['total_claims'] / summary['total_policies']) * 100
print(summary.sort_values(by='total_policies', ascending=False))

# Compare the two main provinces: Gauteng and Western Cape (using full names)
df_prov = df[df['province'].isin(['Gauteng', 'Western Cape'])]
ct, chi2, p_val = run_chi2_test(df_prov, 'province', 'has_claim')

print("\n--- Statistical Test Result ---")
print(f"Chi2 Statistic: {chi2:.4f}")
print(f"p-value: {p_val:.4e}")

if p_val < 0.05:
    print("Decision: REJECT Null Hypothesis (H0). Risk differences are statistically significant.")
else:
    print("Decision: FAIL TO REJECT Null Hypothesis (H0). No statistically significant difference.")



--- Claim Frequency by Province ---
               total_policies  total_claims  claim_rate (%)
province                                                   
Gauteng                393865          1322        0.335648
Western Cape           170796           370        0.216633
KwaZulu-Natal          169781           483        0.284484
North West             143287           349        0.243567
Mpumalanga              52718           128        0.242801
Eastern Cape            30336            50        0.164821
Limpopo                 24836            67        0.269770
Free State               8099            11        0.135819
Northern Cape            6380             8        0.125392

--- Statistical Test Result ---
Chi2 Statistic: 56.0874
p-value: 6.9320e-14
Decision: REJECT Null Hypothesis (H0). Risk differences are statistically significant.


### Interpretation: Hypothesis 1 (Provinces)
I **reject the Null Hypothesis $H_0$** ($p = 6.93 \times 10^{-14} < 0.05$). The difference in claim frequency across provinces is statistically significant. 

* **Takeaway**: Gauteng exhibits a 55% higher claim rate (0.336%) compared to the Western Cape (0.217%). Actuarially, ACIS is warranted to charge higher base premiums for vehicles in Gauteng to reflect this elevated environmental and operational risk.


In [4]:
# Hypothesis 2: H0: There are no risk differences between zip codes.
print("--- Top 5 Postal Codes by Volume ---")
zip_summary = df.groupby('postalcode').agg(
    total_policies=('has_claim', 'count'),
    total_claims=('has_claim', 'sum')
)
zip_summary['claim_rate (%)'] = (zip_summary['total_claims'] / zip_summary['total_policies']) * 100
print(zip_summary.sort_values(by='total_policies', ascending=False).head(5))

# Extract the two most active postal codes to compare
top_zips = zip_summary.sort_values(by='total_policies', ascending=False).index[:2].tolist()
zip_a, zip_b = top_zips[0], top_zips[1]

print(f"\nComparing top two postal codes: {zip_a} vs {zip_b}")
df_zip = df[df['postalcode'].isin([zip_a, zip_b])]
ct_zip, chi2_zip, p_val_zip = run_chi2_test(df_zip, 'postalcode', 'has_claim')

print("\n--- Statistical Test Result ---")
print(f"Chi2 Statistic: {chi2_zip:.4f}")
print(f"p-value: {p_val_zip:.4e}")

if p_val_zip < 0.05:
    print("Decision: REJECT Null Hypothesis (H0). Risk differences between zip codes are statistically significant.")
else:
    print("Decision: FAIL TO REJECT Null Hypothesis (H0). No statistically significant risk difference between these zip codes.")


--- Top 5 Postal Codes by Volume ---
            total_policies  total_claims  claim_rate (%)
postalcode                                              
2000                133498           486        0.364050
122                  49171           210        0.427081
7784                 28585            50        0.174917
299                  25546            67        0.262272
7405                 18518            29        0.156604

Comparing top two postal codes: 2000 vs 122

--- Statistical Test Result ---
Chi2 Statistic: 3.5971
p-value: 5.7882e-02
Decision: FAIL TO REJECT Null Hypothesis (H0). No statistically significant risk difference between these zip codes.


### Interpretation: Hypothesis 2 (Zip Codes - Risk)
I **failed to reject the Null Hypothesis $H_0$** ($p = 0.0579 \ge 0.05$). The difference in claim frequency between postal codes `2000` and `122` is not statistically significant.

* **Takeaway**: Although raw claim rates differ slightly (0.364% vs 0.427%), they are statistically equivalent at a 95% confidence level. ACIS should avoid using these individual zip codes alone as a primary risk segment for premium alterations.


In [5]:
# Hypothesis 3: H0: There is no significant margin (profit) difference between zip codes.
mean_a, mean_b, t_stat, p_val_margin = run_ttest(df, 'postalcode', zip_a, zip_b, 'margin')

print("\n--- Underwriting Margin Statistics ---")
print(f"Postal Code {zip_a} (Control) Average Margin: {mean_a:.2f} Rand")
print(f"Postal Code {zip_b} (Test) Average Margin: {mean_b:.2f} Rand")

print("\n--- Statistical Test Result ---")
print(f"t-Statistic: {t_stat:.4f}")
print(f"p-value: {p_val_margin:.4e}")

if p_val_margin < 0.05:
    print("Decision: REJECT Null Hypothesis (H0). Profitability differences between these zip codes are statistically significant.")
else:
    print("Decision: FAIL TO REJECT Null Hypothesis (H0). No statistically significant difference in underwriting margins.")



--- Underwriting Margin Statistics ---
Postal Code 2000 (Control) Average Margin: -8.11 Rand
Postal Code 122 (Test) Average Margin: -22.86 Rand

--- Statistical Test Result ---
t-Statistic: 1.1639
p-value: 2.4446e-01
Decision: FAIL TO REJECT Null Hypothesis (H0). No statistically significant difference in underwriting margins.


### Interpretation: Hypothesis 3 (Zip Codes - Profit Margin)
I **fail to reject the Null Hypothesis $H_0$** ($p = 0.244 \ge 0.05$). Underwriting margin differences between postal codes `2000` and `122` are not statistically significant.

* **Takeaway**: Profitability margins in both regions are statistically equivalent and both are running at a net loss (negative averages). Rather than adjusting premiums for individual zip codes, ACIS should implement systemic underwriting review in these high-volume urban regions to return to profitability.


In [6]:
# Hypothesis 4: H0: There is no significant risk difference between Women and Men.
print("--- Claim Frequency by Gender ---")
gender_summary = df.groupby('gender').agg(
    total_policies=('has_claim', 'count'),
    total_claims=('has_claim', 'sum')
)
gender_summary['claim_rate (%)'] = (gender_summary['total_claims'] / gender_summary['total_policies']) * 100
print(gender_summary)

# Filter for Male and Female and run Chi-Squared test
df_gender = df[df['gender'].isin(['Male', 'Female'])]
ct_gender, chi2_gender, p_val_gender = run_chi2_test(df_gender, 'gender', 'has_claim')

print("\n--- Statistical Test Result ---")
print(f"Chi2 Statistic: {chi2_gender:.4f}")
print(f"p-value: {p_val_gender:.4e}")

if p_val_gender < 0.05:
    print("Decision: REJECT Null Hypothesis (H0). The risk difference between Women and Men is statistically significant.")
else:
    print("Decision: FAIL TO REJECT Null Hypothesis (H0). No statistically significant risk difference between genders.")


--- Claim Frequency by Gender ---
               total_policies  total_claims  claim_rate (%)
gender                                                     
Female                   6755            14        0.207254
Male                    42817            94        0.219539
Not specified          950526          2680        0.281949

--- Statistical Test Result ---
Chi2 Statistic: 0.0037
p-value: 9.5146e-01
Decision: FAIL TO REJECT Null Hypothesis (H0). No statistically significant risk difference between genders.


### Interpretation: Hypothesis 4 (Gender - Risk)
I **fail to reject the Null Hypothesis $H_0$** ($p = 0.951 \ge 0.05$). The risk difference between Women and Men is not statistically significant.

* **Takeaway**: I observe that the claim frequency for Women is 0.207% and for Men is 0.220%. Actuarially, this difference is practically zero. I recommend that ACIS should not implement gender-based premium adjustments in their new pricing framework, as gender does not act as a distinct risk driver.


In [7]:
# Summary Results Table
results_data = {
    "Hypothesis": [
        "1. Risk Differences Across Provinces",
        "2. Risk Differences Between Zip Codes",
        "3. Margin Differences Between Zip Codes",
        "4. Risk Differences Between Genders"
    ],
    "KPI Type": [
        "Claim Frequency (Categorical)",
        "Claim Frequency (Categorical)",
        "Underwriting Margin (Numerical)",
        "Claim Frequency (Categorical)"
    ],
    "Statistical Test Used": [
        "Chi-Squared contingency test",
        "Chi-Squared contingency test",
        "Welch's t-test (Independent)",
        "Chi-Squared contingency test"
    ],
    "p-value": [
        6.9320e-14,
        5.7882e-02,
        2.4446e-01,
        9.5146e-01
    ],
    "Decision": [
        "REJECT H0 (Statistically Significant)",
        "FAIL TO REJECT H0 (Not Significant)",
        "FAIL TO REJECT H0 (Not Significant)",
        "FAIL TO REJECT H0 (Not Significant)"
    ]
}

# Display results table
df_results = pd.DataFrame(results_data)
pd.set_option('display.max_colwidth', None)
df_results


,Hypothesis,KPI Type,Statistical Test Used,p-value,Decision
0,1. Risk Differences Across Provinces,Claim Frequency (Categorical),Chi-Squared contingency test,6.932000e-14,REJECT H0 (Statistically Significant)
1,2. Risk Differences Between Zip Codes,Claim Frequency (Categorical),Chi-Squared contingency test,5.788200e-02,FAIL TO REJECT H0 (Not Significant)
2,3. Margin Differences Between Zip Codes,Underwriting Margin (Numerical),Welch's t-test (Independent),2.444600e-01,FAIL TO REJECT H0 (Not Significant)
3,4. Risk Differences Between Genders,Claim Frequency (Categorical),Chi-Squared contingency test,9.514600e-01,FAIL TO REJECT H0 (Not Significant)
